# Session 16 lab: maximum likelihood, MAP and the bias-variance trade-off

KLAi | Mathematics for AI and Machine Learning

Today you choose the parameter that makes the data most likely (maximum likelihood estimation, MLE), you show that squared error and cross-entropy come from probability assumptions, you add a prior (maximum a posteriori estimation, MAP) and you watch bias and variance in a simulation.

**How to use this notebook.** Run the cells in order with Shift and Enter. Each step has the same rhythm: **PREDICT**, **RUN**, **READ**, then **CHANGE one thing**. Type your answers in the lines that start with `>`. If a number differs from the slide called "Check your numbers", stop and find the step that went wrong.

**Words and letters.** *Likelihood* p(data | theta): how probable the observed data are for one value theta of the parameter. *Log-likelihood* l(theta): the natural logarithm (ln) of the likelihood. In this course the letter L is the loss, so the likelihood never gets the letter L. *Prior*: what we believe about the parameter before we see the data. *Bias*: the error of the average prediction. *Variance*: how much the prediction moves from one training set to the next.

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks(total=None):
    # total = how many checks this notebook has; a skipped or crashed cell still counts as NOT YET
    k = sum(_results.values()); n = max(total or 0, len(_results))
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(suppress=True, precision=6)

---
## Step 1: Maximum likelihood of a coin, by scanning

A coin is flipped 10 times. 1 means heads and 0 means tails. The unknown chance of heads is the parameter theta (Session 14 called it p). We try nine values of theta and compute the likelihood of this exact sequence of flips for each one. (For 7 heads in any order the probability is 120 times larger. That factor does not depend on theta, so it does not move the peak.)

In [ ]:
flips = np.array([1, 1, 0, 1, 1, 1, 0, 1, 0, 1])      # 1 = heads, 0 = tails
n = len(flips)                                          # number of flips
k = int(flips.sum())                                    # number of heads
theta_grid = np.round(np.linspace(0.1, 0.9, 9), 1)      # the nine values of theta we try: 0.1, 0.2, ... 0.9
print("flips:", n, " heads:", k)

**PREDICT first.** Which of the nine values of theta do you think makes this data most likely? Is it 0.5, 0.7 or 0.9?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# For every theta in theta_grid: how probable is this exact sequence? Each head and each tail contributes one factor (see the slide "By hand: the likelihood of one sequence")
likelihood = None  # your code here

In [ ]:
if likelihood is None:
    print("NOT YET: write your code in the cell above, then run both cells again.")
else:
    from math import comb
    check("likelihood of the exact sequence at theta = 0.7", likelihood[6], 0.0022236, 0.000001)
    check("7 heads in any order at theta = 0.7 is comb(10, 7) times that", comb(n, k) * likelihood[6], 0.2668, 0.0001)
    check("the grid value with the largest likelihood (the MLE)", theta_grid[np.argmax(likelihood)], 0.7, 0.0001)
    for t, lik in zip(theta_grid, likelihood):
        print("theta =", t, "  likelihood =", "%.3g" % lik)

**READ the output.** The likelihoods are tiny numbers (7.29e-08 means 7.29 times 10 to the power -8). Is the largest one at the same theta as k / n? Write k / n here.

> My answer: 

**CHANGE one thing.** Change the flips to all heads: `flips = np.ones(10, dtype=int)` and run Step 1 again. Where is the maximum now? (The grid stops at 0.9, so the maximum lands on its last value although k / n is 1.) What does that say about trusting MLE with very little data? Put the original flips back before Step 2.

> What I saw: 

---
## Step 2: The log-likelihood, and why we use it

A likelihood of many data points is a product of many numbers below 1. Computers cannot hold a product that small, but they can hold a sum of logarithms. The log-likelihood l(theta) uses the natural log, ln, so its unit is the nat.

**PREDICT first.** What will Python print for 0.5 raised to the power 1100 (1,100 fair coin flips, each with probability 0.5)?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
print("0.5 ** 1100 =", 0.5 ** 1100)

In [ ]:
# The log-likelihood is the sum of the natural logs of the same factors (see the log-likelihood slides). Compute it for every theta in theta_grid. Then the log of 0.5**1100 as a sum of 1100 equal logs, with no product
log_likelihood = None  # your code here
log_fair_1100 = None  # your code here

In [ ]:
if log_likelihood is None or log_fair_1100 is None:
    print("NOT YET: write your code in the cell above, then run both cells again.")
else:
    check("log-likelihood of the exact sequence at theta = 0.7", log_likelihood[6], -6.1086, 0.001)
    check("the log-likelihood has its maximum at the same theta", theta_grid[np.argmax(log_likelihood)], 0.7, 0.0001)
    check("log of 0.5**1100 as a sum of logs", log_fair_1100, -762.4619, 0.001)
    print("log-likelihood at theta = 0.7:", round(float(log_likelihood[6]), 4), "nats")
    print("log of 0.5**1100 as a sum:", round(float(log_fair_1100), 2), "nats")

**READ the output.** The product printed 0.0, but the sum of logs is a clear number. Is the largest log-likelihood at the same theta as the largest likelihood?

> My answer: 

---
## Step 3: Maximum likelihood of a Gaussian mean

Eight measurements of the width of a part, in mm. The model is the simplest possible: it predicts the same width mu for every part, so y-hat = mu for every row. We assume each measurement is that prediction plus Gaussian (normal) noise with a known standard deviation sigma = 1 mm.

In [ ]:
y = np.array([9.8, 10.4, 11.2, 9.1, 10.7, 10.1, 11.5, 9.2])   # widths in mm
sigma = 1.0                                                    # known standard deviation of the noise
mu_grid = np.arange(9.0, 11.5001, 0.25)                        # the values of mu we try, 9.00 to 11.50
print("n =", len(y), "  widths:", y)

**PREDICT first.** Which mu will have the largest log-likelihood? Guess before you run, then compare your guess with the average of the eight widths.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# For each mu add up the natural log of the Gaussian density of every y (see the slide "Gaussian noise turns the negative log-likelihood into a squared error plus a constant")
log_lik_mu = None  # your code here

In [ ]:
if log_lik_mu is None:
    print("NOT YET: write your code in the cell above, then run both cells again.")
else:
    best = mu_grid[np.argmax(log_lik_mu)]
    check("best mu on the grid", best, 10.25, 0.0001)
    check("the largest log-likelihood", log_lik_mu.max(), -10.0215, 0.001)
    check("best mu equals the sample mean", y.mean(), best, 0.0001)
    print("best mu:", best, "  sample mean:", y.mean(), "  largest log-likelihood:", round(float(log_lik_mu.max()), 4))
    plt.figure(figsize=(6, 3.2))
    plt.plot(mu_grid, log_lik_mu, "o-", color="black")
    plt.axvline(best, color="#AE2525")
    plt.xlabel("mu (mm)"); plt.ylabel("log-likelihood (nats)"); plt.title("Log-likelihood of the 8 widths"); plt.show()

**READ the output.** The best mu equals the average of the eight widths. Why would the average be the most likely mean?

> My answer: 

**CHANGE one thing.** Change sigma to 2.0 and run Steps 3 and 4 again. Does the best mu move? Does the curve get wider or narrower? Then set sigma back to 1.0 and run both steps again.

> What I saw: 

---
## Step 4: Squared error is the Gaussian log-likelihood in disguise

Compute the sum of squared errors (SSE) for every mu and put it next to the negative log-likelihood.

**PREDICT first.** If we minimise the SSE, will we pick the same mu as when we maximise the log-likelihood?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# For each mu, the sum of the squared gaps between the eight widths and mu
sse = None  # your code here

In [ ]:
if sse is None or log_lik_mu is None:
    print("NOT YET: write your code in Steps 3 and 4, then run the cells again.")
else:
    gap = -log_lik_mu - sse / (2 * sigma**2)          # negative log-likelihood minus SSE / (2 sigma squared)
    check("SSE at the best mu", sse.min(), 5.34, 0.001)
    check("mu with the smallest SSE", mu_grid[np.argmin(sse)], 10.25, 0.0001)
    check("the gap is the same number for every mu", np.ptp(gap), 0.0, 0.000001)
    check("the leftover constant n/2 * ln(2*pi*sigma^2) with sigma = 1", gap.mean(), 7.3515, 0.001)
    print("mu with the smallest SSE:", mu_grid[np.argmin(sse)], "  SSE there:", round(float(sse.min()), 4))
    print("-loglik minus SSE/(2 sigma^2) for every mu:", np.round(gap, 4))

**READ the output.** The last line shows the same number for every mu. That number is n/2 times ln(2 pi sigma squared), 7.3515 for sigma = 1. It does not depend on mu. What does that tell you about the two curves?

> My answer: 

---
## Step 5: Cross-entropy is the Bernoulli log-likelihood

A classifier gives p, its probability of class 1, for four examples. The true labels are 1, 0, 1, 1. The loss uses ln, so it is in nats.

In [ ]:
y_true = np.array([1, 0, 1, 1])
p_pred = np.array([0.8, 0.3, 0.6, 0.9])       # the model's probability of class 1
probs3 = np.array([0.7, 0.2, 0.1])              # a 3-class prediction; the true class is the first one

**PREDICT first.** Will the mean negative log-likelihood of the four labels be bigger than 1, or smaller?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Binary cross-entropy: the mean over the four rows of the negative natural log of the probability given to the true label (see the cross-entropy formula slide). Categorical: the same idea for one true class
bce = None  # your code here
cat_loss = None  # your code here

In [ ]:
if bce is None or cat_loss is None:
    print("NOT YET: write your code in the cell above, then run both cells again.")
else:
    check("binary cross-entropy of the four labels", bce, 0.2990, 0.001)
    check("categorical cross-entropy for true class prob 0.7", cat_loss, 0.3567, 0.001)
    print("binary cross-entropy:", round(float(bce), 4), "nats   categorical loss:", round(float(cat_loss), 4), "nats")

**CHANGE one thing.** Change the model's probability for the second example from 0.3 to 0.9 (a confident wrong answer, since the true label is 0). How much does the cross-entropy rise? Put 0.3 back afterwards.

> What I saw: 

---
## Step 6: MAP: the likelihood times a prior

Back to the coin. We believe a coin is probably near fair, so we give each theta a prior weight: 1, 2, 3, 4, 5, 4, 3, 2, 1 for theta = 0.1 up to 0.9. (Dividing every weight by their total, 25, would turn them into probabilities. That rescales every product by the same factor, so the winner does not change.)

In [ ]:
prior = np.round(5 - 10 * np.abs(theta_grid - 0.5), 6)      # 1, 2, 3, 4, 5, 4, 3, 2, 1
print("prior weights:", prior)

**PREDICT first.** The likelihood alone peaks at theta = 0.7. Will the posterior (likelihood times prior) peak at 0.7, below it, or above it?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# The unnormalised posterior combines the likelihood and the prior weight of each theta (see the MAP slides). The MAP estimate is the theta with the largest one
posterior = None  # your code here

In [ ]:
if posterior is None:
    print("NOT YET: write your code in the cell above, then run both cells again.")
else:
    theta_map = theta_grid[np.argmax(posterior)]
    check("MAP estimate of theta", theta_map, 0.6, 0.0001)
    check("unnormalised posterior at theta = 0.6", posterior[5], 0.0071664, 0.000001)
    print("MAP estimate:", theta_map, "  posterior at that theta:", round(float(posterior.max()), 6))

**Weight decay is the same idea.** A Gaussian prior on a weight w adds an L2 penalty (lambda times the squared weight) to the squared error. Take x = 1, 2, 3 and y = 2.1, 3.9, 6.2 and fit y = w x. The slide "By hand: the penalty pulls the best weight towards 0" gives the penalised answer for this fit.

In [ ]:
x = np.array([1.0, 2.0, 3.0])
y_reg = np.array([2.1, 3.9, 6.2])
lam_values = [0, 1, 5, 14]

In [ ]:
# For every lambda in lam_values, the best w for this fit with the penalty (see the slide "By hand: the penalty pulls the best weight towards 0")
w_ridge = None  # your code here

In [ ]:
if w_ridge is None:
    print("NOT YET: write your code in the cell above, then run both cells again.")
else:
    check("w with no penalty and with lambda = 1, 5, 14", w_ridge, [2.0357, 1.9, 1.5, 1.0179], 0.001)
    print("lambda values:", lam_values)
    print("weights:", np.round(w_ridge, 4))

**READ the output.** As lambda grows, does w get larger or smaller? What would a prior that is very sure that w is near zero do?

> My answer: 

---
## Step 7: Bias and variance in a simulation

We draw 300 fresh noisy training sets of 10 points from a smooth wave (the sine curve; you do not need its formula), fit three polynomials (degree 1, 3 and 9) to each, and measure how the predictions behave. Session 12's degree-9 curve that chased the noise is the high-variance case.

**PREDICT first.** Which degree will have the largest bias, and which the largest variance?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
import warnings
warnings.filterwarnings("ignore")             # a degree-9 fit on 10 points makes numpy complain; that is the point of the picture
rng = np.random.default_rng(16)               # a fixed seed so everyone gets the same numbers
x_train = np.linspace(0, 1, 10)               # 10 training inputs, the same in every repeat
x_test = np.linspace(0.05, 0.95, 10)          # 10 places where we look at the predictions
true_f = lambda x: np.sin(2 * np.pi * x)      # the true curve, a smooth wave (we pretend nature knows it)
noise = 0.3                                   # standard deviation of the noise in each y
repeats = 300                                 # 300 fresh training sets
degrees = [1, 3, 9]                           # the three models: a line, a cubic, a degree-9 polynomial
preds = {d: np.zeros((repeats, len(x_test))) for d in degrees}
for r in range(repeats):
    y_train = true_f(x_train) + rng.normal(0, noise, size=len(x_train))   # one new noisy training set
    for d in degrees:
        coef = np.polyfit(x_train, y_train, d)                            # fit a polynomial of degree d
        preds[d][r] = np.polyval(coef, x_test)                           # its predictions at the 10 test inputs
bias2 = {d: float(np.mean((preds[d].mean(axis=0) - true_f(x_test)) ** 2)) for d in degrees}   # squared bias
variance = {d: float(np.mean(preds[d].var(axis=0))) for d in degrees}                         # variance
for d in degrees:
    print("degree", d, " bias squared", round(bias2[d], 4), " variance", round(variance[d], 4))

In [ ]:
# Expected test error for each degree: the three parts on the slide "Expected error splits into bias squared, variance and noise"
expected_error = None  # your code here

In [ ]:
if expected_error is None:
    print("NOT YET: write your code in the cell above, then run both cells again.")
else:
    check("expected error, degree 1", expected_error[1], 0.3189, 0.001)
    check("expected error, degree 3", expected_error[3], 0.1270, 0.001)
    check("expected error, degree 9", expected_error[9], 0.9148, 0.001)
    check("degree 9 has the largest variance", max(variance, key=variance.get), 9, 0.0001)
    check("degree 1 has the largest bias squared", max(bias2, key=bias2.get), 1, 0.0001)
    print({d: round(v, 4) for d, v in expected_error.items()})
    fig, axes = plt.subplots(1, 3, figsize=(10, 2.8), sharey=True)
    xx = np.linspace(0, 1, 100)
    for ax, d in zip(axes, degrees):
        for r in range(30):                                  # draw 30 of the 300 fitted curves
            ax.plot(x_test, preds[d][r], color="#8C8C8C", alpha=0.4)
        ax.plot(xx, true_f(xx), color="#AE2525", lw=2)
        ax.set_title("degree " + str(d)); ax.set_ylim(-2, 2)
    plt.show()

**READ the output.** Which degree has the smallest expected error? Is it the simplest model, the most flexible one, or the one in between?

> My answer: 

**CHANGE one thing.** Change `noise = 0.3` to `noise = 0.6` in the simulation cell and run it and the cell after it again. Which degree gets worse fastest?

> What I saw: 

---
## Save your work

Run every cell again from the top (Runtime, Restart and run all), check the last line says all checks pass, then save the notebook.

In [ ]:
show_checks(23)